### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="cardiotocography",
    dataset_year="2010",
    domain_str="medical & healthcare",
    # Data Source
    dataset_source="UCI",
    original_dataset_source_download_link="10.24432/C51S4N",
    download_description="""
wget https://archive.ics.uci.edu/static/public/193/cardiotocography.zip \
&& unzip cardiotocography.zip && rm cardiotocography.zip \
&& mkdir -p local-data-warehouse/cardiotocography \
&& mv CTG.xls local-data-warehouse/cardiotocography
""",
    # References
    academic_reference_bibtex="""@misc{campos2010cardiotocography,
  author       = {Campos, D. and Bernardes, J.},
  title        = {{Cardiotocography}},
  year         = {2000},
  howpublished = {UCI Machine Learning Repository},
  note         = {{DOI}: https://doi.org/10.24432/C51S4N}
}
""",
    academic_reference_bibtex_key="campos2010cardiotocography",
    license="CC BY 4.0",
    data_tags=["Non-IID", "Grouped"],
    curation_comments="""
We use the data from UCI and the 3 class problem of predicting NSP as it is more medical relevant.

- We transform the file names into patient IDs to indicate the sub-group of samples recorded from the same patient.
- The raw data has dates and the start and end index of the recording. We drop these as they are not relevant for the predictive task and would not be available for test samples.
- We drop all other columns that represent the label.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="NSP",
    problem_type="multiclass_classification",
    objective_metric_name="log_loss",
    # For classification
    stratify_on="NSP",
    # For grouped data
    group_on="patient_id",
    group_labels="per_sample",
)

## Preprocessing

In [2]:
import pandas as pd
import uuid


# Read excel file
df = pd.read_excel(dataset_mold.path / "CTG.xls", sheet_name="Raw Data")
print("Loaded data shape:", df.shape)

# -- Remove rows that are not data rows from Excel sheet
# Drop first row, which is missing in the Excel sheet
df = df.drop(index=0).reset_index(drop=True)
# Drop last three rows as they are also corrupted
df = df.drop(index=df.index[-3:]).reset_index(drop=True)


df['FileName'] = (
    df['FileName']
    .str.replace(r'\.[^.]+$', '', regex=True)
    .pipe(lambda s: s.where(s.str.fullmatch(r'S\d+'),
                           s.str.replace(r'(_?\d+)$', '', regex=True)))
)
# Create mapping: molecule -> random string id
mapping = {val: uuid.uuid4().hex[:12] for val in df["FileName"].unique()}
df["patient_id"] = df["FileName"].map(mapping)

df = df.drop(columns=[
    "FileName",
    # Duplicat of LB
    "LBE",
    # Non-feature information
    "b",
    "e",
    "Date",
    "SegFile",
    # Classes
    "A", "B", "C", "D", "E",
    "AD","DE","LD","FS","SUSP","CLASS",


])

as_cat_dtype = ["patient_id", "NSP"]
df[as_cat_dtype] = df[as_cat_dtype].astype("category")

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

Loaded data shape: (2130, 40)


"## Data Checks

In [3]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 2,126
Columns: 24
Use sampling: False (sample size: 2,126)
Get row duplicates (staged, merged)...
Using top-10 columns for initial filtering: ['MLTV', 'patient_id', 'Width', 'Variance', 'Min', 'Mean', 'FM', 'Median', 'Mode', 'ALTV']
Rows remaining as candidates after top-10 filter: 37 (of 2,126)

#### Duplicate Report
Total duplicate rows: 14 (0.66% of dataset)
Duplicate rows ignoring target: 16 (0.75% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [4]:
# Sample Rows
df_head

,LB,AC,FM,UC,ASTV,MSTV,ALTV,MLTV,DL,DS,DP,DR,Width,Min,Max,Nmax,Nzeros,Mode,Mean,Median,Variance,Tendency,NSP,patient_id
0,133.0,2.0,12.0,3.0,46.0,1.1,0.0,15.4,2.0,0.0,0.0,0.0,69.0,95.0,164.0,5.0,0.0,139.0,135.0,138.0,9.0,0.0,1.0,9ed7407309f0
1,125.0,0.0,1.0,8.0,62.0,1.7,0.0,1.1,7.0,0.0,0.0,0.0,72.0,68.0,140.0,5.0,0.0,130.0,116.0,125.0,29.0,1.0,1.0,d89588c7073d
2,131.0,5.0,3.0,5.0,60.0,2.1,0.0,0.9,6.0,0.0,1.0,0.0,90.0,78.0,168.0,8.0,0.0,133.0,127.0,132.0,21.0,0.0,1.0,86cf95440ea4
3,131.0,8.0,0.0,4.0,29.0,1.3,0.0,4.5,0.0,0.0,0.0,0.0,89.0,82.0,171.0,8.0,0.0,143.0,145.0,145.0,9.0,1.0,1.0,93d4991b7750
4,125.0,0.0,0.0,8.0,64.0,1.3,0.0,2.6,7.0,0.0,1.0,0.0,77.0,78.0,155.0,4.0,0.0,114.0,111.0,114.0,7.0,0.0,1.0,d89588c7073d


In [5]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,NSP,category,0.0,0.0,3.0,"1.0, 2.0, 3.0"
1,patient_id,category,0.0,0.0,176.0,"58fd7a892599, 4c45fda319d2, 77b837108ca3, d609ae36ae69, 2fb0f3273025, d66167f4716b, bcf9703ea64d, d9f13721a442, e7642a9dade6, d89588c7073d"
2,LB,float64,0.0,0.0,48.0,"133.0, 130.0, 122.0, 138.0, 125.0, 128.0, 120.0, 144.0, 142.0, 132.0"
3,AC,float64,0.0,0.0,22.0,"0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0"
4,FM,float64,0.0,0.0,96.0,"0.0, 1.0, 2.0, 3.0, 4.0, 6.0, 7.0, 5.0, 8.0, 10.0"
5,UC,float64,0.0,0.0,19.0,"0.0, 3.0, 4.0, 1.0, 2.0, 5.0, 6.0, 7.0, 8.0, 9.0"
6,ASTV,float64,0.0,0.0,75.0,"60.0, 58.0, 65.0, 63.0, 64.0, 61.0, 51.0, 62.0, 22.0, 25.0"
7,MSTV,float64,0.0,0.0,57.0,"0.8, 1.3, 0.5, 0.4, 0.7, 0.9, 0.6, 1.2, 1.5, 1.0"
8,ALTV,float64,0.0,0.0,87.0,"0.0, 1.0, 2.0, 5.0, 4.0, 3.0, 8.0, 6.0, 12.0, 7.0"
9,MLTV,float64,0.0,0.0,249.0,"0.0, 6.7, 7.1, 5.2, 6.5, 9.5, 6.8, 5.6, 8.5, 7.2"


In [6]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
LB,2126.0,133.303857,9.840844,106.0,160.0
AC,2126.0,2.722484,3.560850,0.0,26.0
FM,2126.0,7.241298,37.125309,0.0,564.0
UC,2126.0,3.659925,2.847094,0.0,23.0
ASTV,2126.0,46.990122,17.192814,12.0,87.0
MSTV,2126.0,1.332785,0.883241,0.2,7.0
ALTV,2126.0,9.846660,18.396880,0.0,91.0
MLTV,2126.0,8.187629,5.628247,0.0,50.7
DL,2126.0,1.570085,2.499229,0.0,16.0
DS,2126.0,0.003293,0.057300,0.0,1.0


In [7]:
# Categorical Feature Statistics
cat_stats

value  count    pct
column     rank                            
NSP        1              1.0   1655  77.85
           2              2.0    295  13.88
           3              3.0    176   8.28
patient_id 1     58fd7a892599     54   2.54
           2     4c45fda319d2     53   2.49
           3     77b837108ca3     40   1.88
           4     d609ae36ae69     37   1.74
           5     2fb0f3273025     36   1.69

In [8]:
# Target Distribution
target_df

,count,pct
NSP,,
1.0,1655,77.85
2.0,295,13.88
3.0,176,8.28


## Task Curation

In [9]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(
    dataset=df,
    group_on=task_mold.group_on,
    time_on=task_mold.time_on,
    group_labels=task_mold.group_labels,
)
print(f"Recommended splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended splits: n_repeats=10, n_splits=3, test_size=None


In [10]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry import curation_recommendations

# -- For Grouped Non-IID data
splits = curation_recommendations.get_recommended_grouped_splits(
    dataset=df,
    n_repeats=n_repeats,
    n_splits=n_splits,
    group_on=task_mold.group_on,
    test_size=none_or_test_size,
    stratify_on=task_mold.stratify_on,
    group_labels=task_mold.group_labels,
    show_splits=True,
    target_on=task_mold.target_column_name,
)

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits.",
    splits=splits,
)

Using Stratified Grouped splits.
Using label-per-sample grouped splits.


Repeat 0, Fold 0:
            Train N: 1254, Test N: 872
            Target Distribution:
            	Train target distribution: {1.0: 0.8157894736842105, 2.0: 0.11244019138755981, 3.0: 0.07177033492822966}
            	Test target distribution: {1.0: 0.7247706422018348, 2.0: 0.17660550458715596, 3.0: 0.09862385321100918}
            Group Distribution patient_id:
            	Train: 119
            	Test: 57
            
Repeat 0, Fold 1:
            Train N: 1544, Test N: 582
            Target Distribution:
            	Train target distribution: {1.0: 0.7519430051813472, 2.0: 0.15155440414507773, 3.0: 0.09650259067357513}
            	Test target distribution: {1.0: 0.8487972508591065, 2.0: 0.10481099656357389, 3.0: 0.04639175257731959}
            Group Distribution patient_id:
            	Train: 118
            	Test: 58
            
Repeat 0, Fold 2:
            Train N: 1454, Test N: 672
            Target Distribution:
            	Train target distribution: {1.0: 0.77441540

## Export

In [11]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
Saving curated container to cardiotocography/019d7392-12c2-7a87-991c-13a7246555d4
019d7392-12c2-7a87-991c-13a7246555d4
dfc2f3a1c5e8685180111b7d1720d0ab8f8d95d26a0b495b77ca910b4d203c04
